# שלב 09 — Link Prediction ושיפור הרשת

מחקר טוב לא רק מאבחן אלא גם מציע פתרון. כאן בודקים שיטות לחיזוי קשתות חסרות (Common Neighbors, Jaccard, Adamic-Adar, Resource Allocation, Preferential Attachment, ו‑Node2Vec), ואז משתמשים ב‑embeddings כדי להציע חיבורים חדשים ולבדוק אם הם משפרים את עמידות הרשת.

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas numpy networkx matplotlib seaborn python-bidi scikit-learn

In [ ]:
from pathlib import Path
import pickle, json
import pandas as pd
import numpy as np
import networkx as nx
import random
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
BASE = ROOT / "public_transport_network_notebooks"
GRAPH_DIR = BASE / "outputs" / "02_graph_construction"
EMB_CSV = BASE / "outputs" / "08_graph_learning_node_embeddings" / "embeddings_df.csv"
OUT_DIR = BASE / "outputs" / "09_link_prediction_and_network_improvement"
FIG_DIR = BASE / "figures" / "09_link_prediction_and_network_improvement"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TEST_FRAC, SEED, TOP_K_SUGGEST = 0.10, 42, 20
print("OUT_DIR:", OUT_DIR)

In [ ]:
import re
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from bidi.algorithm import get_display


def _fix(t):
    """מסדר טקסט עברי לתצוגה נכונה (bidi). אנגלית ומספרים נשארים כמו שהם."""
    if isinstance(t, str) and any(0x590 <= ord(c) <= 0x5FF for c in t):
        return get_display(t)
    return t


import matplotlib.text as _mt
if not getattr(_mt.Text, "_bidi", False):
    _orig = _mt.Text.set_text
    def _set(self, s):
        if isinstance(s, str) and getattr(self, "_disp", None) == s:
            return _orig(self, s)
        f = _fix(s)
        if isinstance(f, str):
            self._disp = f
        return _orig(self, f)
    _mt.Text.set_text = _set
    _mt.Text._bidi = True

sns.set_theme(style="whitegrid", font_scale=1.1)
matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("עברית בגרפים מופעלת")

## טעינה והכנת דאטה

עובדים על הרכיב הקשור הגדול. מסתירים 10% מהקשתות כ'חיוביות' לבדיקה, ודוגמים זוגות לא‑מחוברים כ'שליליות'. המודלים צריכים להבדיל בין השניים.

In [ ]:
with open(GRAPH_DIR / "graph_undirected.pkl", "rb") as f:
    G_full = pickle.load(f)
G = G_full.subgraph(max(nx.connected_components(G_full), key=len)).copy()

embeddings = {}
if EMB_CSV.exists():
    emb_df = pd.read_csv(EMB_CSV, encoding="utf-8-sig")
    emb_cols = [c for c in emb_df.columns if c.startswith("e")]
    for _, row in emb_df.iterrows():
        try:
            key = str(int(float(row["stop_id"])))
        except (ValueError, TypeError):
            key = str(row["stop_id"])
        embeddings[key] = row[emb_cols].values.astype(float)
print(f"גרף: {G.number_of_nodes():,} צמתים | Embeddings: {len(embeddings)}")

rng = random.Random(SEED)
edges = list(G.edges())
rng.shuffle(edges)
n_test = int(len(edges) * TEST_FRAC)
test_pos = edges[:n_test]
G_train = G.copy(); G_train.remove_edges_from(test_pos)

existing = {frozenset((u, v)) for u, v in G.edges()}
nodes = list(G.nodes())
test_neg, seen, attempts = [], set(), 0
while len(test_neg) < n_test and attempts < n_test * 50:
    attempts += 1
    u, v = rng.choice(nodes), rng.choice(nodes)
    if u == v:
        continue
    key = frozenset((u, v))
    if key in existing or key in seen:
        continue
    seen.add(key); test_neg.append((u, v))
print(f"{len(test_pos)} חיוביות, {len(test_neg)} שליליות לבדיקה")

## שיטות קלאסיות ל-Link Prediction

כל שיטה נותנת ציון לכל זוג, ואנחנו מודדים AUC-ROC (יכולת ההפרדה בין קשת אמיתית לזוג אקראי).

In [ ]:
pairs = test_pos + test_neg
labels = [1]*len(test_pos) + [0]*len(test_neg)

def scores_of(gen):
    return {(u, v): s for u, v, s in gen}

methods = {
    "Common Neighbors": scores_of(nx.common_neighbor_centrality(G_train, pairs)),
    "Jaccard": scores_of(nx.jaccard_coefficient(G_train, pairs)),
    "Adamic-Adar": scores_of(nx.adamic_adar_index(G_train, pairs)),
    "Resource Allocation": scores_of(nx.resource_allocation_index(G_train, pairs)),
    "Preferential Attachment": scores_of(nx.preferential_attachment(G_train, pairs)),
}

results = {}
for name, sc in methods.items():
    s = [sc.get((u, v), sc.get((v, u), 0.0)) for u, v in pairs]
    results[name] = {"auc": round(roc_auc_score(labels, s), 4),
                     "average_precision": round(average_precision_score(labels, s), 4)}
    print(f"{name}: AUC={results[name]['auc']:.4f}")

## Link Prediction מבוסס Node2Vec

לכל זוג בונים מאפיין Hadamard (מכפלת הווקטורים) ומאמנים Logistic Regression. בדרך כלל זו השיטה החזקה ביותר.

In [ ]:
if embeddings:
    def feat(u, v):
        eu, ev = embeddings.get(str(u)), embeddings.get(str(v))
        return None if eu is None or ev is None else eu * ev
    X, yv = [], []
    for (u, v), lbl in zip(pairs, labels):
        f = feat(u, v)
        if f is not None:
            X.append(f); yv.append(lbl)
    X, yv = np.array(X), np.array(yv)
    Xtr, Xte, ytr, yte = train_test_split(X, yv, test_size=0.3, random_state=SEED, stratify=yv)
    sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    clf = LogisticRegression(class_weight="balanced", max_iter=500).fit(Xtr, ytr)
    prob = clf.predict_proba(Xte)[:, 1]
    results["Node2Vec + LR"] = {"auc": round(roc_auc_score(yte, prob), 4),
                                "average_precision": round(average_precision_score(yte, prob), 4)}
    print(f"Node2Vec + LR: AUC={results['Node2Vec + LR']['auc']:.4f}")

results_df = pd.DataFrame([{"method": k, **v} for k, v in results.items()])
results_df.to_csv(OUT_DIR / "link_prediction_results.csv", index=False, encoding="utf-8-sig")
results_df

## גרף: השוואת השיטות

In [ ]:
dfp = pd.DataFrame([(k, v["auc"]) for k, v in results.items()], columns=["method", "AUC"]).sort_values("AUC")
colors = ["#dc2626" if v < 0.55 else "#2563eb" if v < 0.65 else "#16a34a" for v in dfp["AUC"]]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(dfp["method"], dfp["AUC"], color=colors)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1, label="Baseline (0.5)")
ax.set_xlabel("AUC-ROC"); ax.set_title("השוואת שיטות Link Prediction"); ax.legend()
for i, (_, row) in enumerate(dfp.iterrows()):
    ax.text(row["AUC"] + 0.002, i, f"{row['AUC']:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "method_comparison_auc_bar.png", dpi=150)
plt.show()